In [ ]:
"""
=============================================================================
PIPELINE ETL: CONSTRUÇÃO DA BASE BRUTA DE PESSOA FÍSICA (SCR/BACEN)
Revisão Quant Sênior: Processamento Out-of-Core, Prevenção de Survivorship Bias,
Otimização de Memória (Downcast/Category) e Parsing Explícito de Datas.
Autor: Itaiguara de Oliveira Bezerra
=============================================================================
"""

import glob
import logging
import os
import unicodedata
import zipfile
from typing import Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ==============================================================================
# CONFIGURAÇÕES GERAIS E CONSTANTES
# ==============================================================================
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

PASTA_ZIPS = "dados_scr/"
SAIDA_PARQUET = "painel_scr_pf_bruto.parquet"

TERMOS_SALDO = [
    "carteira", "vencido", "inadimplencia", "inadimplente", 
    "ativo", "saldo", "valor", "a_vencer", "prejuizo", "provisao"
]

COLUNAS_CATEGORICAS = ["uf", "cnae_ocupacao", "modalidade", "instituicao", "porte", "indexador"]

# ==============================================================================
# FUNÇÕES DE ENGENHARIA DE DADOS
# ==============================================================================
def normalizar_nome_coluna(col: str) -> str:
    """Remove acentos, caracteres especiais e padroniza para snake_case minúsculo."""
    col_str = str(col).strip()
    nfkd = unicodedata.normalize("NFKD", col_str)
    sem_acento = "".join([c for c in nfkd if not unicodedata.combining(c)])
    return (
        sem_acento.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace(".", "")
    )

def gerar_base_pf_bruta_completa(pasta: str = PASTA_ZIPS) -> None:
    """
    Processa múltiplos arquivos ZIP do Bacen via arquitetura Out-of-Core, 
    realizando filtros estritos, downcasting numérico e exportação incremental em Parquet.
    """
    arquivos_zip = sorted(glob.glob(os.path.join(pasta, "*.zip")))

    if not arquivos_zip:
        logger.error(f"Nenhum arquivo .zip encontrado na pasta: '{pasta}'")
        return None

    print("=" * 95)
    print(f"ETL SCR/BACEN: INGESTÃO INTEGRAL OUT-OF-CORE DE {len(arquivos_zip)} ARQUIVOS COMPACTADOS")
    print("=" * 95)

    writer: Optional[pq.ParquetWriter] = None
    total_arquivos_processados = 0
    total_arquivos_rejeitados = 0
    total_registros_validados = 0

    for arq_zip in arquivos_zip:
        nome_zip = os.path.basename(arq_zip)
        print(f"\n>>> Processando arquivo compactado: {nome_zip}")

        try:
            with zipfile.ZipFile(arq_zip, "r") as z:
                for nome_csv in z.namelist():
                    if nome_csv.lower().endswith(".csv"):
                        with z.open(nome_csv) as f:
                            try:
                                df_chunk = pd.read_csv(f, sep=";", encoding="utf-8", low_memory=False)
                            except UnicodeDecodeError:
                                f.seek(0)
                                df_chunk = pd.read_csv(f, sep=";", encoding="latin1", low_memory=False)

                            df_chunk.columns = [normalizar_nome_coluna(c) for c in df_chunk.columns]

                            # 1. Filtro Estrito PF
                            col_cli = next((c for c in ["cliente", "tipo_cliente"] if c in df_chunk.columns), None)
                            if col_cli:
                                mask_pf = df_chunk[col_cli].astype(str).str.strip().str.upper().isin(["PF", "PESSOA FISICA", "PESSOA FÍSICA"])
                                df_chunk = df_chunk[mask_pf]
                            elif "cnae_ocupacao" in df_chunk.columns:
                                df_chunk = df_chunk[df_chunk["cnae_ocupacao"].notna()]
                            else:
                                logger.warning(f"Lote REJEITADO: '{nome_csv}'. Sem coluna de identificação PF.")
                                total_arquivos_rejeitados += 1
                                continue

                            if len(df_chunk) == 0:
                                continue

                            # 2. Parsing de Data Explícito
                            if "data_base" in df_chunk.columns:
                                df_chunk["data_base"] = pd.to_datetime(df_chunk["data_base"], format="%Y-%m-%d", errors="coerce")

                            # 3. Otimização Numérica (Downcast para float32)
                            cols_monetarias_encontradas = []
                            for col in df_chunk.columns:
                                if any(termo in col for termo in TERMOS_SALDO):
                                    if df_chunk[col].dtype == "object":
                                        serie_limpa = df_chunk[col].astype(str).str.strip().str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
                                        serie_num = pd.to_numeric(serie_limpa, errors="coerce")
                                    else:
                                        serie_num = pd.to_numeric(df_chunk[col], errors="coerce")
                                    
                                    if "carteira" in col:
                                        df_chunk[col] = serie_num.astype(np.float32)
                                    else:
                                        df_chunk[col] = serie_num.fillna(0.0).astype(np.float32)
                                    
                                    cols_monetarias_encontradas.append(col)

                            # 4. Correção do Viés de Sobrevivência (Mapeamento de Exposição Total)
                            if cols_monetarias_encontradas:
                                df_chunk["carteira_total"] = df_chunk[cols_monetarias_encontradas].sum(axis=1, skipna=True)
                                # Expurga estritamente apenas se a exposição total for zero/nula
                                df_chunk = df_chunk[df_chunk["carteira_total"].notna() & (df_chunk["carteira_total"] > 0)].copy()

                            # 5. Otimização de Tipos Categóricos
                            for col_cat in COLUNAS_CATEGORICAS:
                                if col_cat in df_chunk.columns:
                                    df_chunk[col_cat] = df_chunk[col_cat].astype("category")

                            # 6. Gravação Incremental em Disco (Evita OOM - Out of Memory)
                            if len(df_chunk) > 0:
                                table = pa.Table.from_pandas(df_chunk)
                                
                                if writer is None:
                                    writer = pq.ParquetWriter(SAIDA_PARQUET, table.schema, compression="snappy")
                                
                                writer.write_table(table)
                                
                                total_arquivos_processados += 1
                                total_registros_validados += len(df_chunk)
                                print(f"   -> {nome_csv}: {len(df_chunk):,} registros consolidados e salvos.")

        except Exception as e:
            logger.error(f"Erro ao processar '{nome_zip}': {e}")
            continue

    if writer:
        writer.close()

    print("\n" + "=" * 95)
    print("BASE BRUTA PF CONSOLIDADA (ARQUITETURA OUT-OF-CORE)")
    print("=" * 95)
    print(f"Total de registros validados e salvos: {total_registros_validados:,}")
    print(f"Total de arquivos processados: {total_arquivos_processados}")
    print(f"Total de arquivos rejeitados: {total_arquivos_rejeitados}")
    print("=" * 95)

if __name__ == "__main__":
    gerar_base_pf_bruta_completa()